# 🚗 UA-DETRAC Trafik Veri Seti ile YOLO11 Araç Tespiti (Uçtan Uca Pipeline)
### Bitirme Projesi - Gerçek Trafik Kameraları ile Yüksek Başarımlı Model Eğitimi

Bu notebook, Google Drive'ınızdaki **`DETRAC-Images.zip`** arşivini otomatik olarak okur, etiketleri işler, **YOLO11** modelini T4 GPU ile eğitir ve sonuçları paketler.

### 📌 UA-DETRAC Veri Seti Nedir?
* **Kökeni:** Albany Üniversitesi (University at Albany) ve Trafik İzleme Laboratuvarı tarafından geliştirilmiş, dünya çapında en çok kabul gören trafik gözetleme (traffic surveillance) benchmark veri setidir.
* **Kamera Konumu:** Şehir içi üst geçit ve MOBESE kameralarından çekilmiş gerçek yol görüntüleridir.
* **Koşullar:** Gündüz, gece, yağmurlu, bulutlu ve yoğun trafik gibi gerçek yol senaryolarını içerir.
* **Hedef Araç Sınıfları:** `car` (otomobil/van), `motorcycle` (motosiklet), `bus` (otobüs) ve `truck` (kamyon/ağır vasıta).

👉 **Tüm adımları sırayla (1'den 7'ye) çalıştırmanız yeterlidir.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## Adım 1: Donanım Kontrolü ve Kütüphane Kurulumu
GPU hızlandırıcısının devrede olduğunu kontrol edip Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [ ]:
import os
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! T4 GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL! Lütfen üst menüden: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçip Kaydet deyin.')

!pip install -q ultralytics
print('✅ Ultralytics ve PyTorch ortamı hazır!')


## Adım 2: Proje Deposunu ve XML Etiketlerini Çekme
GitHub üzerindeki repoyu ve içerisinde hazır bulunan XML koordinat etiketlerini çekiyoruz.


In [ ]:
import os

repo_name = 'federated-traffic-learning'
if not os.path.exists(repo_name):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    %cd federated-traffic-learning
    !git pull

print('✅ Proje dizini ve XML etiket paketi hazır:', os.getcwd())


## Adım 3: Google Drive'daki DETRAC-Images.zip'i Bağlama ve YOLO Formatına Dönüştürme
Drive'a attığınız **`DETRAC-Images.zip`** arşivini otomatik olarak bulur, projedeki XML etiketleriyle eşleştirir ve gereksiz kopyalama yapmadan doğrudan eğitim veri setini hazırlar.


In [ ]:
import os
import glob
import zipfile
import random
import xml.etree.ElementTree as ET
from google.colab import drive

# 1. Google Drive'ı bağla
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# 2. DETRAC-Images.zip dosyasını Drive içinde ara
print('🔍 Google Drive içinde DETRAC-Images.zip aranıyor...')
search_patterns = [
    '/content/drive/MyDrive/**/DETRAC-Images.zip',
    '/content/drive/MyDrive/**/DETRAC*.zip'
]
found_images_zips = []
for p in search_patterns:
    matches = glob.glob(p, recursive=True)
    if matches:
        found_images_zips.extend(matches)
        break

if not found_images_zips:
    raise FileNotFoundError('❌ Google Drive içerisinde DETRAC-Images.zip bulunamadı! Lütfen dosyanın Drive hesabınızda olduğundan emin olun.')

images_zip_path = found_images_zips[0]
print(f'✅ Drive\'da DETRAC-Images.zip bulundu: {images_zip_path}')

# 3. XML etiket arşivini aç
xml_zip_path = 'detrac_xml_annotations.zip'
annot_dir = '/content/DETRAC_annotations'
if not os.path.exists(annot_dir):
    print('📦 XML etiketleri açılıyor...')
    !unzip -q -o "{xml_zip_path}" -d "{annot_dir}"
    print('✅ XML etiketleri hazır!')

# 4. Çıkış dizinlerini oluştur
out_dataset_dir = '/content/ua_detrac_dataset'
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(out_dataset_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(out_dataset_dir, 'labels', split), exist_ok=True)

CLASS_MAP = {'car': 0, 'van': 0, 'bus': 2, 'others': 3}
IMG_W, IMG_H = 960.0, 540.0

def parse_xml(xml_p):
    tree = ET.parse(xml_p)
    root = tree.getroot()
    seq = root.attrib.get('name', os.path.splitext(os.path.basename(xml_p))[0]).strip()
    frames = {}
    for fe in root.iter('frame'):
        n = fe.attrib.get('num')
        if not n:
            continue
        fnum = int(n)
        boxes = []
        tl = fe.find('target_list')
        if tl is not None:
            for tgt in tl.findall('target'):
                bx = tgt.find('box')
                att = tgt.find('attribute')
                if bx is None or att is None:
                    continue
                vt = att.attrib.get('vehicle_type', 'car').lower()
                if vt not in CLASS_MAP:
                    continue
                cid = CLASS_MAP[vt]
                try:
                    l = float(bx.attrib.get('left', 0))
                    t = float(bx.attrib.get('top', 0))
                    w = float(bx.attrib.get('width', 0))
                    h = float(bx.attrib.get('height', 0))
                except:
                    continue
                if w < 5 or h < 5:
                    continue
                xc = max(0.0, min(1.0, (l + w / 2.0) / IMG_W))
                yc = max(0.0, min(1.0, (t + h / 2.0) / IMG_H))
                wn = max(0.001, min(1.0, w / IMG_W))
                hn = max(0.001, min(1.0, h / IMG_H))
                boxes.append((cid, xc, yc, wn, hn))
        frames[fnum] = boxes
    return seq, frames

# 5. Dizi listelerini hazırla
train_xml_dir = os.path.join(annot_dir, 'train')
test_xml_dir = os.path.join(annot_dir, 'test')
train_xml_files = sorted(glob.glob(f'{train_xml_dir}/*.xml'))
test_xml_files = sorted(glob.glob(f'{test_xml_dir}/*.xml'))

random.seed(42)
random.shuffle(train_xml_files)
val_count = max(6, int(len(train_xml_files) * 0.15))
val_xmls = train_xml_files[:val_count]
train_xmls = train_xml_files[val_count:]

print(f'\n🎬 Video Dizileri -> Eğitim: {len(train_xmls)}, Doğrulama: {len(val_xmls)}, Test: {len(test_xml_files)}')
print('🚀 DETRAC-Images.zip içinden en kaliteli kareler seçilip etiketleniyor (1-2 dakika sürer)...')

zf = zipfile.ZipFile(images_zip_path, 'r')
zip_names = set(zf.namelist())

def extract_split(xml_list, split_name, step):
    img_dir = os.path.join(out_dataset_dir, 'images', split_name)
    lbl_dir = os.path.join(out_dataset_dir, 'labels', split_name)
    count = 0
    for xp in xml_list:
        seq_name, f_dict = parse_xml(xp)
        for fnum in sorted(f_dict.keys())[::step]:
            boxes = f_dict[fnum]
            if not boxes:
                continue
            key = f'DETRAC-Images/{seq_name}/img{fnum:05d}.jpg'
            if key not in zip_names:
                key = f'{seq_name}/img{fnum:05d}.jpg'
                if key not in zip_names:
                    continue
            
            fname = f'{seq_name}_img{fnum:05d}'
            with open(os.path.join(img_dir, f'{fname}.jpg'), 'wb') as img_f:
                img_f.write(zf.read(key))
            with open(os.path.join(lbl_dir, f'{fname}.txt'), 'w', encoding='utf-8') as lbl_f:
                for b in boxes:
                    lbl_f.write(f'{b[0]} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f} {b[4]:.6f}\n')
            count += 1
    return count

# Örnekleme adımları: Her 20. kare
train_cnt = extract_split(train_xmls, 'train', step=20)
val_cnt = extract_split(val_xmls, 'val', step=20)
test_cnt = extract_split(test_xml_files, 'test', step=30)
zf.close()

# 6. data.yaml oluştur
yaml_text = f"""# UA-DETRAC YOLO Dataset Configuration
path: {out_dataset_dir}
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""
data_yaml_path = os.path.join(out_dataset_dir, 'data.yaml')
with open(data_yaml_path, 'w', encoding='utf-8') as yf:
    yf.write(yaml_text.strip() + '\n')

print('\n' + '=' * 60)
print('✅ UA-DETRAC VERİ SETİ BAŞARIYLA HAZIRLANDI!')
print(f' • Eğitim Görselleri:    {train_cnt:,}')
print(f' • Doğrulama Görselleri: {val_cnt:,}')
print(f' • Test Görselleri:      {test_cnt:,}')
print(f' • Hazırlanan data.yaml: {data_yaml_path}')
print('=' * 60)


## Adım 4: Veri Seti Detaylı Analiz Raporu 📊
Hazırlanan veri setindeki tüm araç sınıflarını (`car`, `motorcycle`, `bus`, `truck`), etiketli araç sayılarını ve oranlarını ekrana döker.


In [ ]:
import os
import glob
from collections import Counter

dataset_root = '/content/ua_detrac_dataset'
class_names_map = {0: 'car', 1: 'motorcycle', 2: 'bus', 3: 'truck'}

print('=' * 68)
print('         📊 UA-DETRAC VERİ SETİ DETAYLI ANALİZ RAPORU')
print('=' * 68)
print(f'📁 Veri Seti Dizini: {dataset_root}\n')

total_overall_boxes = 0
overall_class_counts = Counter()

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(dataset_root, 'images', split)
    lbl_dir = os.path.join(dataset_root, 'labels', split)
    
    imgs = glob.glob(f'{img_dir}/*.jpg')
    lbls = glob.glob(f'{lbl_dir}/*.txt')
    
    class_counts = Counter()
    split_boxes = 0
    for lf in lbls:
        try:
            with open(lf, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if parts:
                        cls_id = int(float(parts[0]))
                        class_counts[cls_id] += 1
                        split_boxes += 1
        except:
            pass
            
    overall_class_counts.update(class_counts)
    total_overall_boxes += split_boxes
    
    print(f'📌 [{split.upper()} KÜMESİ]')
    print(f'   • Görsel Sayısı: {len(imgs):,}')
    print(f'   • Etiketli Araç Kutusu (Bounding Box): {split_boxes:,}')
    print('   • Sınıf Dağılımı:')
    for cid in sorted(class_counts.keys()):
        cname = class_names_map.get(cid, f'class_{cid}')
        pct = (class_counts[cid] / split_boxes * 100) if split_boxes > 0 else 0.0
        print(f'      - ID {cid:02d} ({cname:<12}): {class_counts[cid]:>7,} adet  (%{pct:>5.1f})')
    print('-' * 68)

print('\n🎯 [GENEL TOPLAM]')
print(f'   • Toplam Etiketlenen Araç Sayısı: {total_overall_boxes:,}')
for cid in sorted(overall_class_counts.keys()):
    cname = class_names_map.get(cid, f'class_{cid}')
    pct = (overall_class_counts[cid] / total_overall_boxes * 100) if total_overall_boxes > 0 else 0.0
    print(f'   • Sınıf {cid} ({cname:<12}): {overall_class_counts[cid]:>8,} adet  (%{pct:>5.1f})')
print('=' * 68)


## Adım 5: YOLO11 Modelini Eğitme (Training) 🚀
45.000+ gerçek araç verisiyle **YOLO11 Nano** modelini T4 GPU üzerinde eğitiyoruz.


In [ ]:
import os
import torch
from ultralytics import YOLO

device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'🚀 Model Eğitimi Başlatılıyor... Donanım: {device_arg} (GPU: {torch.cuda.is_available()})')

# YOLO11 Nano modelini yüklüyoruz
model = YOLO('yolo11n.pt')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,
    project='runs/train',
    name='yolo11n_uad_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## Adım 6: Model Başarım Testi ve Gerçek Fotoğraf Kıyaslaması 📸
Bağımsız test kümesi üzerinde kesinlik (mAP) değerlerini ölçer ve özel test fotoğrafı üzerinde eğitim öncesi vs. sonrası tespit farkını yan yana görselleştirir.


In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

# 1. Test Kümesi Değerlendirmesi
metrics = model.val(data='/content/ua_detrac_dataset/data.yaml', split='test')
print('\n' + '=' * 50)
print('      MODEL BAŞARI METRİKLERİ (UA-DETRAC)')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)

# 2. Gerçek Trafik Test Fotoğrafı Üzerinde Kıyaslama
custom_photo = 'test_samples/user_test_photo.jpg'
trained_weights = 'runs/train/yolo11n_uad_exp/weights/best.pt'

if os.path.exists(custom_photo) and os.path.exists(trained_weights):
    print('\n📸 Özel test fotoğrafı kıyaslaması yapılıyor...')
    base_model = YOLO('yolo11n.pt')
    base_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='base_pred', exist_ok=True)
    
    trained_model = YOLO(trained_weights)
    trained_model.predict(source=custom_photo, conf=0.25, save=True, project='runs/detect', name='trained_pred', exist_ok=True)
    
    base_imgs = glob.glob('runs/detect/base_pred/*.*')
    trained_imgs = glob.glob('runs/detect/trained_pred/*.*')
    
    if base_imgs and trained_imgs:
        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(Image.open(base_imgs[0]))
        axes[0].set_title('Eğitim Öncesi: Base YOLO11n (Pretrained)', fontsize=15)
        axes[0].axis('off')
        
        axes[1].imshow(Image.open(trained_imgs[0]))
        axes[1].set_title('Eğitim Sonrası: UA-DETRAC ile Eğitilmiş Model', fontsize=15)
        axes[1].axis('off')
        
        plt.tight_layout()
        out_comp = 'runs/train/yolo11n_uad_exp/custom_test_comparison.jpg'
        plt.savefig(out_comp, bbox_inches='tight', dpi=150)
        print(f'✅ Karşılaştırma görseli kaydedildi: {out_comp}')
        plt.show()


## Adım 7: Sonuçları Paketleme ve Otomatik İndirme 📦
Eğitilen `best.pt` model ağırlıklarını, grafiklerini ve karşılaştırma görselini tek bir `yolo11_results.zip` dosyası haline getirip bilgisayarınıza indirir.


In [ ]:
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11n_uad_exp'
best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Eğitilmiş en başarılı model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('\n🎉 TEBRİKLER! Tüm eğitim ve test işlemleri başarıyla tamamlandı!')
print('👉 İnen "yolo11_results.zip" dosyasını masaüstündeki proje klasörünüze atıp asistanınıza bildirebilirsiniz.')
